# CapFinch Online Platform — Synthetic Data Generator

Builds a small, internally-consistent fake e-commerce dataset (6 tables) to develop and test
the analytics / KPI pipeline **before** the real store launches.

**Two anchor keys**
- `transaction_id` — PK of `orders` (one row per completed sale → revenue, AOV)
- `customer_id` — PK of `customers` (one row per person → repeat rate, demographics)

They join via `customer_id` (FK inside `orders`).

**Relationship chain**
```
customers ─┬─ sessions ── events
           └─ orders ── order_items ── products
```

**KPI targets baked into the generator (by construction)**
| Metric | Target |
|---|---|
| Conversion rate (orders / sessions) | ~2% |
| Cart abandonment (1 − completed/carts) | ~68% |
| Repeat purchase (customers with ≥2 orders) | ~20% |

Everything is produced as a pandas `DataFrame` first (no CSV yet). Scale up later by raising
`N_CUSTOMERS` — the ratios above are preserved. A commented CSV-export cell is at the bottom.


In [ ]:
# If the libraries are not installed, uncomment the next line:
# %pip install faker pandas numpy
#h

import random

import numpy as np
import pandas as pd
from faker import Faker

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
fake = Faker("en_US")
Faker.seed(SEED)

pd.set_option("display.max_columns", None)
print("Libraries ready.")

Libraries ready.


## Config & KPI targets

`N_CUSTOMERS` is the single knob for dataset size. Repeat buyers get a 2nd order, so total
orders ≈ `N_CUSTOMERS × (1 + repeat_rate)`. Session and cart counts are derived from the orders
so the conversion (~2%) and abandonment (~68%) targets hold at any scale.


In [5]:
# ---- Scale (raise this to grow every table; KPI ratios are preserved) ----
N_CUSTOMERS = 15
N_PRODUCTS = 12

# ---- KPI targets ----
TARGET_CONVERSION = 0.02   # orders / total sessions
TARGET_ABANDONMENT = 0.68  # 1 - completed_carts / carts_created
TARGET_REPEAT = 0.20       # share of customers with >= 2 orders

# ---- Categorical vocabularies ----
GENDERS = ["female", "male", "nonbinary"]
ACQ_SOURCES = ["organic", "mailchimp", "social", "referral"]
DEVICES = ["mobile", "desktop", "tablet"]
PAYMENT_METHODS = ["card", "apple_pay", "paypal"]
LANDING_PAGES = ["/", "/new-arrivals", "/sale", "/collections/best-sellers", "/product"]
EVENT_TYPES = ["page_view", "product_view", "add_to_cart", "checkout_start", "purchase"]

repeat_customers = round(N_CUSTOMERS * TARGET_REPEAT)
print(f"Config: {N_CUSTOMERS} customers ({repeat_customers} repeat buyers), {N_PRODUCTS} products")

Config: 15 customers (3 repeat buyers), 12 products


## Table 4 — `products`  [PK: product_id]
Built first because orders/order_items and events reference it.


In [6]:
CATALOG = [
    ("Linen Button Shirt", "Apparel"),
    ("Merino Wool Sweater", "Apparel"),
    ("Slim Fit Chinos", "Apparel"),
    ("Silk Scarf", "Accessories"),
    ("Leather Belt", "Accessories"),
    ("Canvas Tote Bag", "Accessories"),
    ("Suede Loafers", "Footwear"),
    ("Leather Sneakers", "Footwear"),
    ("Scented Soy Candle", "Home"),
    ("Ceramic Mug Set", "Home"),
    ("Vitamin C Serum", "Beauty"),
    ("Matte Lip Balm", "Beauty"),
]


def price_band(p):
    if p < 25:
        return "<$25"
    if p <= 75:
        return "$25-75"
    return "$75+"


products = []
for i, (name, cat) in enumerate(CATALOG[:N_PRODUCTS], start=1):
    price = round(np.random.uniform(12, 140), 2)
    products.append({
        "product_id": f"PROD{i:04d}",
        "product_name": name,
        "category": cat,
        "price": price,
        "price_band": price_band(price),
        "cost": round(price * np.random.uniform(0.4, 0.6), 2),
        "stock_on_hand": int(np.random.randint(0, 200)),
    })

products_df = pd.DataFrame(products)
products_df

,product_id,product_name,category,price,price_band,cost,stock_on_hand
0,PROD0001,Linen Button Shirt,Apparel,59.94,$25-75,35.37,106
1,PROD0002,Merino Wool Sweater,Apparel,111.80,$75+,58.07,121
2,PROD0003,Slim Fit Chinos,Apparel,31.97,$25-75,13.16,87
3,PROD0004,Silk Scarf,Accessories,54.71,$25-75,23.45,130
4,PROD0005,Leather Belt,Accessories,14.63,<$25,8.69,157
5,PROD0006,Canvas Tote Bag,Accessories,39.18,$25-75,17.10,20
6,PROD0007,Suede Loafers,Footwear,91.04,$75+,47.55,88
7,PROD0008,Leather Sneakers,Footwear,49.28,$25-75,25.74,169
8,PROD0009,Scented Soy Candle,Home,17.97,<$25,10.69,14
9,PROD0010,Ceramic Mug Set,Home,70.38,$25-75,39.20,50


## Table 1 — `customers`  [PK: customer_id]
`total_orders` is assigned here (repeat buyers get 2). `first_order_date` is left blank now and
filled in from the actual orders once they exist.


In [7]:
def age_band(a):
    if a <= 24:
        return "18-24"
    if a <= 34:
        return "25-34"
    if a <= 44:
        return "35-44"
    return "45+"


# order-count per customer: repeat buyers get 2 orders, everyone else gets 1
order_counts = [2] * repeat_customers + [1] * (N_CUSTOMERS - repeat_customers)
random.shuffle(order_counts)

customers = []
for i in range(1, N_CUSTOMERS + 1):
    age = int(np.random.randint(18, 66))
    customers.append({
        "customer_id": f"CUST{i:04d}",
        "email": fake.unique.email(),
        "age": age,
        "age_band": age_band(age),
        "gender": random.choice(GENDERS),
        "state": fake.state_abbr(),
        "zip": fake.zipcode(),
        "signup_date": fake.date_between(start_date="-2y", end_date="-30d"),
        "acquisition_source": random.choice(ACQ_SOURCES),
        "first_order_date": pd.NaT,        # filled after orders are built
        "total_orders": order_counts[i - 1],
    })

customers_df = pd.DataFrame(customers)
customers_df

,customer_id,email,age,age_band,gender,state,zip,signup_date,acquisition_source,first_order_date,total_orders
0,CUST0001,johnsonjoshua@example.org,56,45+,female,DC,97031,2024-11-02,organic,NaT,1
1,CUST0002,garzaanthony@example.org,35,35-44,female,IL,30996,2025-08-12,mailchimp,NaT,1
2,CUST0003,hoffmanjennifer@example.net,21,18-24,female,ND,55488,2025-01-24,organic,NaT,1
3,CUST0004,lisa02@example.net,42,35-44,nonbinary,NE,45098,2025-03-05,mailchimp,NaT,1
4,CUST0005,susanrogers@example.org,31,25-34,nonbinary,MS,13177,2025-05-01,referral,NaT,1
5,CUST0006,cassandra07@example.net,26,25-34,female,DE,50116,2024-10-17,referral,NaT,1
6,CUST0007,maria95@example.net,43,35-44,nonbinary,ID,92850,2024-10-10,social,NaT,1
7,CUST0008,kendragalloway@example.org,19,18-24,female,VI,13739,2025-05-16,mailchimp,NaT,2
8,CUST0009,xreid@example.org,37,35-44,nonbinary,MN,47067,2025-01-16,referral,NaT,1
9,CUST0010,jacqueline19@example.net,45,45+,male,GA,70511,2026-01-16,social,NaT,1


## Tables 2 & 3 — `orders` and `order_items`
Each order expands from a customer's `total_orders`. Line items drive `order_total` (= sum of
`line_total`) and `item_count`. Every order also creates its **converting session** so the
funnel stays consistent.


In [8]:
order_rows, oi_rows, sess_rows = [], [], []
order_seq = sess_seq = 0

product_ids = products_df["product_id"].tolist()
price_lookup = products_df.set_index("product_id")["price"].to_dict()

for _, cust in customers_df.iterrows():
    n = int(cust["total_orders"])
    base = pd.to_datetime(cust["signup_date"]) + pd.Timedelta(days=int(np.random.randint(1, 60)))
    dates = sorted(base + pd.to_timedelta(np.random.randint(0, 400, size=n), unit="D"))

    for k in range(n):
        order_seq += 1
        sess_seq += 1
        tid = f"TXN{order_seq:05d}"
        sid = f"SESS{sess_seq:06d}"
        order_dt = pd.to_datetime(dates[k]) + pd.Timedelta(minutes=int(np.random.randint(2, 40)))

        # line items
        n_items = int(np.random.randint(1, 5))
        chosen = np.random.choice(product_ids, size=n_items, replace=False)
        order_total, total_qty = 0.0, 0
        for pid in chosen:
            qty = int(np.random.randint(1, 4))
            unit = float(price_lookup[pid])
            line = round(qty * unit, 2)
            order_total += line
            total_qty += qty
            oi_rows.append({
                "order_item_id": f"OI{len(oi_rows) + 1:06d}",
                "transaction_id": tid,
                "product_id": str(pid),
                "quantity": qty,
                "unit_price": unit,
                "line_total": line,
            })
        order_total = round(order_total, 2)

        order_rows.append({
            "transaction_id": tid,
            "customer_id": cust["customer_id"],
            "session_id": sid,
            "order_date": order_dt,
            "order_total": order_total,
            "item_count": total_qty,
            "payment_method": random.choice(PAYMENT_METHODS),
            "payment_status": np.random.choice(["authorized", "declined"], p=[0.95, 0.05]),
            "shipping_state": cust["state"],
            "discount_amount": round(order_total * np.random.choice([0, 0.05, 0.10], p=[0.7, 0.2, 0.1]), 2),
            "is_first_order": k == 0,
        })

        # converting session tied to this order
        sess_rows.append({
            "session_id": sid,
            "customer_id": cust["customer_id"],
            "session_start": order_dt - pd.Timedelta(minutes=int(np.random.randint(3, 30))),
            "device": random.choice(DEVICES),
            "traffic_source": cust["acquisition_source"],
            "landing_page": random.choice(LANDING_PAGES),
            "reached_cart": True,
            "converted": True,
        })

orders_df = pd.DataFrame(order_rows)
order_items_df = pd.DataFrame(oi_rows)
print(f"orders={len(orders_df)}  order_items={len(order_items_df)}")
orders_df

orders=18  order_items=48


,transaction_id,customer_id,session_id,order_date,order_total,item_count,payment_method,payment_status,shipping_state,discount_amount,is_first_order
0,TXN00001,CUST0001,SESS000001,2025-06-30 00:18:00,385.81,9,card,authorized,DC,38.58,True
1,TXN00002,CUST0002,SESS000002,2026-03-29 00:22:00,373.04,6,paypal,authorized,IL,0.00,True
2,TXN00003,CUST0003,SESS000003,2025-04-20 00:12:00,303.45,6,card,authorized,ND,0.00,True
3,TXN00004,CUST0004,SESS000004,2025-07-18 00:23:00,592.66,8,paypal,authorized,NE,29.63,True
4,TXN00005,CUST0005,SESS000005,2026-06-26 00:14:00,537.15,9,paypal,authorized,MS,53.72,True
5,TXN00006,CUST0006,SESS000006,2025-04-25 00:29:00,319.48,5,card,authorized,DE,0.00,True
6,TXN00007,CUST0007,SESS000007,2025-09-01 00:38:00,244.14,2,apple_pay,authorized,ID,12.21,True
7,TXN00008,CUST0008,SESS000008,2025-09-22 00:03:00,157.86,4,card,authorized,VI,0.00,True
8,TXN00009,CUST0008,SESS000009,2026-01-06 00:18:00,278.76,6,apple_pay,authorized,VI,0.00,False
9,TXN00010,CUST0009,SESS000010,2026-01-15 00:04:00,176.16,8,card,authorized,MN,0.00,True


In [9]:
# order_items preview
order_items_df.head(15)

,order_item_id,transaction_id,product_id,quantity,unit_price,line_total
0,OI000001,TXN00001,PROD0003,3,31.97,95.91
1,OI000002,TXN00001,PROD0001,2,59.94,119.88
2,OI000003,TXN00001,PROD0005,2,14.63,29.26
3,OI000004,TXN00001,PROD0010,2,70.38,140.76
4,OI000005,TXN00002,PROD0012,1,122.07,122.07
5,OI000006,TXN00002,PROD0003,1,31.97,31.97
6,OI000007,TXN00002,PROD0006,1,39.18,39.18
7,OI000008,TXN00002,PROD0001,3,59.94,179.82
8,OI000009,TXN00003,PROD0004,3,54.71,164.13
9,OI000010,TXN00003,PROD0011,1,60.96,60.96


## Table 5 — `sessions`  [PK: session_id]
The converting sessions already exist (one per order). Here we add **abandoned-cart** and
**browse-only** sessions so the totals hit conversion ~2% and abandonment ~68%:

- `total_carts = orders / (1 − 0.68)` → abandoned = carts − orders
- `total_sessions = orders / 0.02` → browse-only = sessions − carts

Some non-converting sessions are anonymous (`customer_id = None`).


In [10]:
n_orders = len(orders_df)
total_carts = round(n_orders / (1 - TARGET_ABANDONMENT))
abandoned_carts = max(total_carts - n_orders, 0)
total_sessions = round(n_orders / TARGET_CONVERSION)
browse_sessions = max(total_sessions - total_carts, 0)

cust_ids = customers_df["customer_id"].tolist()


def make_session(reached_cart, converted):
    global sess_seq
    sess_seq += 1
    # ~40% of non-converting traffic is a known customer, the rest is anonymous
    cust_id = random.choice(cust_ids) if random.random() < 0.4 else None
    return {
        "session_id": f"SESS{sess_seq:06d}",
        "customer_id": cust_id,
        "session_start": pd.to_datetime(fake.date_time_between(start_date="-1y", end_date="now")),
        "device": random.choice(DEVICES),
        "traffic_source": random.choice(ACQ_SOURCES),
        "landing_page": random.choice(LANDING_PAGES),
        "reached_cart": reached_cart,
        "converted": converted,
    }


for _ in range(abandoned_carts):
    sess_rows.append(make_session(reached_cart=True, converted=False))
for _ in range(browse_sessions):
    sess_rows.append(make_session(reached_cart=False, converted=False))

sessions_df = pd.DataFrame(sess_rows).sort_values("session_start").reset_index(drop=True)
print(f"sessions={len(sessions_df)}  carts={total_carts}  abandoned={abandoned_carts}  browse={browse_sessions}")
sessions_df.head(15)

sessions=900  carts=56  abandoned=38  browse=844


,session_id,customer_id,session_start,device,traffic_source,landing_page,reached_cart,converted
0,SESS000003,CUST0003,2025-04-19 23:46:00.000000,tablet,organic,/sale,True,True
1,SESS000006,CUST0006,2025-04-25 00:00:00.000000,tablet,referral,/new-arrivals,True,True
2,SESS000012,CUST0011,2025-06-24 00:05:00.000000,tablet,mailchimp,/,True,True
3,SESS000001,CUST0001,2025-06-30 00:00:00.000000,tablet,organic,/collections/best-sellers,True,True
4,SESS000004,CUST0004,2025-07-17 23:55:00.000000,tablet,mailchimp,/sale,True,True
5,SESS000687,NaN,2025-08-24 18:33:01.931578,desktop,mailchimp,/collections/best-sellers,False,False
6,SESS000141,NaN,2025-08-25 03:21:14.705616,mobile,mailchimp,/,False,False
7,SESS000732,NaN,2025-08-25 06:56:49.212350,desktop,organic,/collections/best-sellers,False,False
8,SESS000041,NaN,2025-08-25 12:54:29.583459,desktop,organic,/new-arrivals,True,False
9,SESS000092,NaN,2025-08-26 23:29:37.407471,mobile,mailchimp,/,False,False


### Back-fill `first_order_date`
Now that orders exist, set each customer's first order date from their earliest order.


In [11]:
first_orders = orders_df.groupby("customer_id")["order_date"].min()
customers_df["first_order_date"] = pd.to_datetime(customers_df["customer_id"].map(first_orders)).dt.date
customers_df

,customer_id,email,age,age_band,gender,state,zip,signup_date,acquisition_source,first_order_date,total_orders
0,CUST0001,johnsonjoshua@example.org,56,45+,female,DC,97031,2024-11-02,organic,2025-06-30,1
1,CUST0002,garzaanthony@example.org,35,35-44,female,IL,30996,2025-08-12,mailchimp,2026-03-29,1
2,CUST0003,hoffmanjennifer@example.net,21,18-24,female,ND,55488,2025-01-24,organic,2025-04-20,1
3,CUST0004,lisa02@example.net,42,35-44,nonbinary,NE,45098,2025-03-05,mailchimp,2025-07-18,1
4,CUST0005,susanrogers@example.org,31,25-34,nonbinary,MS,13177,2025-05-01,referral,2026-06-26,1
5,CUST0006,cassandra07@example.net,26,25-34,female,DE,50116,2024-10-17,referral,2025-04-25,1
6,CUST0007,maria95@example.net,43,35-44,nonbinary,ID,92850,2024-10-10,social,2025-09-01,1
7,CUST0008,kendragalloway@example.org,19,18-24,female,VI,13739,2025-05-16,mailchimp,2025-09-22,2
8,CUST0009,xreid@example.org,37,35-44,nonbinary,MN,47067,2025-01-16,referral,2026-01-15,1
9,CUST0010,jacqueline19@example.net,45,45+,male,GA,70511,2026-01-16,social,2026-06-02,1


## Table 6 — `events`  [PK: event_id]
Funnel detail per session. Every session gets a `page_view` + some `product_view`s; sessions
that reached the cart add `add_to_cart` → `checkout_start`, and converters end with `purchase`.


In [12]:
event_rows = []


def add_event(session, etype, t, pid=None):
    event_rows.append({
        "event_id": f"EVT{len(event_rows) + 1:07d}",
        "session_id": session["session_id"],
        "event_type": etype,
        "product_id": pid,
        "event_time": t,
    })


for _, s in sessions_df.iterrows():
    t = pd.to_datetime(s["session_start"])
    add_event(s, "page_view", t)

    for _ in range(int(np.random.randint(1, 4))):
        t += pd.Timedelta(seconds=int(np.random.randint(20, 180)))
        add_event(s, "product_view", t, random.choice(product_ids))

    if s["reached_cart"]:
        t += pd.Timedelta(seconds=int(np.random.randint(20, 120)))
        add_event(s, "add_to_cart", t, random.choice(product_ids))
        t += pd.Timedelta(seconds=int(np.random.randint(20, 120)))
        add_event(s, "checkout_start", t)
        if s["converted"]:
            t += pd.Timedelta(seconds=int(np.random.randint(20, 120)))
            add_event(s, "purchase", t)

events_df = pd.DataFrame(event_rows)
print(f"events={len(events_df)}")
events_df.head(15)

events=2808


,event_id,session_id,event_type,product_id,event_time
0,EVT0000001,SESS000003,page_view,NaN,2025-04-19 23:46:00
1,EVT0000002,SESS000003,product_view,PROD0001,2025-04-19 23:47:06
2,EVT0000003,SESS000003,product_view,PROD0009,2025-04-19 23:49:04
3,EVT0000004,SESS000003,add_to_cart,PROD0004,2025-04-19 23:50:18
4,EVT0000005,SESS000003,checkout_start,NaN,2025-04-19 23:51:17
5,EVT0000006,SESS000003,purchase,NaN,2025-04-19 23:52:28
6,EVT0000007,SESS000006,page_view,NaN,2025-04-25 00:00:00
7,EVT0000008,SESS000006,product_view,PROD0012,2025-04-25 00:02:13
8,EVT0000009,SESS000006,add_to_cart,PROD0006,2025-04-25 00:03:02
9,EVT0000010,SESS000006,checkout_start,NaN,2025-04-25 00:03:40


## Overview, KPI check & integrity
Confirm the six DataFrames, verify the KPI targets, and assert the numeric relationships hold.


In [13]:
tables = {
    "customers": customers_df,
    "orders": orders_df,
    "order_items": order_items_df,
    "products": products_df,
    "sessions": sessions_df,
    "events": events_df,
}

print("Table shapes")
for name, df in tables.items():
    print(f"  {name:12s} rows={len(df):5d}  cols={len(df.columns)}")

# --- KPI check ---
conversion = len(orders_df) / len(sessions_df)
carts = int(sessions_df["reached_cart"].sum())
abandonment = 1 - sessions_df["converted"].sum() / carts
repeat = (customers_df["total_orders"] >= 2).mean()

print("\nKPIs (actual vs target)")
print(f"  Conversion rate : {conversion:6.2%}  (target ~2%)")
print(f"  Cart abandonment: {abandonment:6.2%}  (target ~68%)")
print(f"  Repeat purchase : {repeat:6.2%}  (target ~20%)")

# --- Integrity checks ---
assert (order_items_df["line_total"] == (order_items_df["quantity"] * order_items_df["unit_price"]).round(2)).all()
recon = order_items_df.groupby("transaction_id")["line_total"].sum().round(2)
assert np.allclose(recon.values, orders_df.set_index("transaction_id").loc[recon.index, "order_total"].values)
assert orders_df["customer_id"].isin(customers_df["customer_id"]).all()
assert order_items_df["product_id"].isin(products_df["product_id"]).all()
print("\nIntegrity checks passed: line totals, order totals, and FK references all consistent.")

Table shapes
  customers    rows=   15  cols=11
  orders       rows=   18  cols=11
  order_items  rows=   48  cols=6
  products     rows=   12  cols=7
  sessions     rows=  900  cols=8
  events       rows= 2808  cols=5

KPIs (actual vs target)
  Conversion rate :  2.00%  (target ~2%)
  Cart abandonment: 67.86%  (target ~68%)
  Repeat purchase : 20.00%  (target ~20%)

Integrity checks passed: line totals, order totals, and FK references all consistent.


## Export to CSV (optional)
Everything lives as DataFrames above. Uncomment to write them to a `data/` folder when ready.


In [14]:
# import os
# os.makedirs("data", exist_ok=True)
# for name, df in tables.items():
#     df.to_csv(f"data/{name}.csv", index=False)
# print("Wrote:", ", ".join(f"data/{n}.csv" for n in tables))